## **Connect Google Drive**

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Extract dataset**

In [5]:
import os
import zipfile

# Google Drive-এ রাখা ZIP file-এর location
zip_path = "/content/drive/MyDrive/Colab Notebooks/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis.zip"

# Dataset extract করার folder
extract_path = "/content/lung_data"

os.makedirs(extract_path, exist_ok=True)

# ZIP file extract করা
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

# Dataset-এর folder structure দেখা
for root, dirs, files in os.walk(extract_path):

    if dirs or files:
        print("Folder:", root)
        print("Subfolders:", dirs)
        print("Number of files:", len(files))
        print()

Dataset extracted successfully!
Folder: /content/lung_data
Subfolders: ['An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis']
Number of files: 0

Folder: /content/lung_data/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis
Subfolders: ['Lung Cancer Diagnosis Dataset']
Number of files: 0

Folder: /content/lung_data/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis/Lung Cancer Diagnosis Dataset
Subfolders: ['Dataset']
Number of files: 0

Folder: /content/lung_data/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis/Lung Cancer Diagnosis Dataset/Dataset
Subfolders: ['Malignant cases', 'Bengin cases', 'Normal cases']
Number of files: 0

Folder: /content/lung_data/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis/Lung Cancer Diagnosis Dataset/Dataset/Malignant cases
Subfolders: []
Number of files: 1050

Folder: /content/lung_data/An Image Dataset of Advanced Imaging Technique

## **Split dataset (70:15:15)**

In [6]:
import os
import shutil
import random

# Original dataset folder
SOURCE_DIR = "/content/lung_data/An Image Dataset of Advanced Imaging Techniques for Lung Cancer Diagnosis/Lung Cancer Diagnosis Dataset/Dataset"

# Save the split dataset in Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/Dataset/Lung_Cancer_Split_Dataset_V4"

# Fixed seed for reproducibility
SEED = 42

# Original folder names and new class names
classes = {
    "Malignant cases": "Malignant",
    "Bengin cases": "Benign",
    "Normal cases": "Normal"
}

# Remove previous split dataset if it exists
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

# Create train, valid and test folders
for split in ["train", "valid", "test"]:
    for class_name in classes.values():
        os.makedirs(
            os.path.join(OUTPUT_DIR, split, class_name),
            exist_ok=True
        )

# Randomly split images class by class
for folder_name, class_name in classes.items():

    source_path = os.path.join(SOURCE_DIR, folder_name)

    images = [
        f for f in os.listdir(source_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp"))
    ]

    # Random shuffle
    rng = random.Random(SEED)
    rng.shuffle(images)

    total = len(images)

    # 70:15:15 split
    train_end = int(total * 0.70)
    valid_end = int(total * 0.85)

    split_images = {
        "train": images[:train_end],
        "valid": images[train_end:valid_end],
        "test": images[valid_end:]
    }

    # Copy images into respective folders
    for split, files in split_images.items():
        for file_name in files:
            shutil.copy2(
                os.path.join(source_path, file_name),
                os.path.join(OUTPUT_DIR, split, class_name, file_name)
            )

    print(
        class_name,
        "Train:", len(split_images["train"]),
        "Valid:", len(split_images["valid"]),
        "Test:", len(split_images["test"])
    )

print("\nOriginal split dataset saved successfully!")

Malignant Train: 735 Valid: 157 Test: 158
Benign Train: 326 Valid: 70 Test: 71
Normal Train: 291 Valid: 62 Test: 63

Original split dataset saved successfully!


## **Augmentation + Balance dataset + Save**

In [7]:
import os
import random
import shutil
from PIL import Image, ImageEnhance

# Fixed seed for reproducibility
SEED = 42
random.seed(SEED)

# Original split dataset (source)
SPLIT_DIR = "/content/drive/MyDrive/Dataset/Lung_Cancer_Split_Dataset_V4"

# New augmented dataset (destination)
OUTPUT_DIR = "/content/drive/MyDrive/Dataset/Lung_Cancer_Augmented_Dataset_V4"

# Copy the original split dataset to a new folder
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

shutil.copytree(SPLIT_DIR, OUTPUT_DIR)

# Target number of training images per class
TARGET_COUNT = 735

# Image extension
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# Training folder location
TRAIN_DIR = os.path.join(OUTPUT_DIR, "train")


# Function to apply augmentation
def augment_image(image):

    # Convert image to RGB
    image = image.convert("RGB")

    # 1. Small random rotation
    angle = random.uniform(-8, 8)
    image = image.rotate(angle)

    # 2. Small random zoom
    width, height = image.size

    zoom = random.uniform(0.90, 1.0)

    crop_width = int(width * zoom)
    crop_height = int(height * zoom)

    left = (width - crop_width) // 2
    top = (height - crop_height) // 2

    image = image.crop((
        left,
        top,
        left + crop_width,
        top + crop_height
    ))

    image = image.resize((width, height))

    # 3. Slight brightness adjustment
    brightness = random.uniform(0.90, 1.10)

    image = ImageEnhance.Brightness(image).enhance(brightness)

    return image


# Balance Benign and Normal classes
for class_name in ["Benign", "Normal"]:

    class_dir = os.path.join(TRAIN_DIR, class_name)

    # Remove previously generated augmented images
    for file_name in os.listdir(class_dir):

        if file_name.startswith("aug_"):
            os.remove(os.path.join(class_dir, file_name))

    # Get original training images only
    original_images = [
        f for f in os.listdir(class_dir)
        if f.lower().endswith(IMAGE_EXTENSIONS)
    ]

    current_count = len(original_images)

    # Calculate how many new images are needed
    images_needed = TARGET_COUNT - current_count

    print(f"\nProcessing {class_name}")
    print("Original images:", current_count)
    print("Additional images needed:", images_needed)

    # Generate augmented images
    for i in range(images_needed):

        # Randomly select an original image
        selected_image = random.choice(original_images)

        image_path = os.path.join(class_dir, selected_image)

        # Open original image
        with Image.open(image_path) as img:

            # Apply augmentation
            augmented = augment_image(img)

            # Save new augmented image
            save_path = os.path.join(
                class_dir,
                f"aug_{i+1:04d}.jpg"
            )

            augmented.save(save_path, quality=95)

    print(class_name, "balancing completed!")


# Final dataset verification
print("\n========== FINAL DATASET COUNT ==========")

for split in ["train", "valid", "test"]:

    print(f"\n{split.upper()} DATASET")

    for class_name in ["Malignant", "Benign", "Normal"]:

        class_dir = os.path.join(OUTPUT_DIR, split, class_name)

        count = len([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(IMAGE_EXTENSIONS)
        ])

        print(class_name, ":", count)

print("\nBalanced dataset saved successfully in Google Drive!")


Processing Benign
Original images: 326
Additional images needed: 409
Benign balancing completed!

Processing Normal
Original images: 291
Additional images needed: 444
Normal balancing completed!

========== FINAL DATASET COUNT ==========

TRAIN DATASET
Malignant : 735
Benign : 735
Normal : 735

VALID DATASET
Malignant : 157
Benign : 70
Normal : 62

TEST DATASET
Malignant : 158
Benign : 71
Normal : 63

Balanced dataset saved successfully in Google Drive!


In [8]:

import time
from PIL import Image

print("\n========== VERIFYING DATASET ==========")

MAX_RETRIES = 30
WAIT_SECONDS = 10

verified = False

for attempt in range(1, MAX_RETRIES + 1):

    all_valid = True
    total_images = 0

    for split in ["train", "valid", "test"]:

        for class_name in ["Malignant", "Benign", "Normal"]:

            folder = os.path.join(OUTPUT_DIR, split, class_name)

            files = [
                f for f in os.listdir(folder)
                if f.lower().endswith(IMAGE_EXTENSIONS)
            ]

            total_images += len(files)

            for filename in files:

                filepath = os.path.join(folder, filename)

                try:
                    with Image.open(filepath) as img:
                        img.verify()

                except Exception:
                    all_valid = False
                    print("Verification failed:", filepath)
                    break

            if not all_valid:
                break

        if not all_valid:
            break

    if all_valid:
        verified = True
        break

    print(f"Attempt {attempt}/{MAX_RETRIES}: Waiting 10 seconds...")
    time.sleep(WAIT_SECONDS)


if verified:

    print("\n====================================")
    print("DATASET CREATED AND VERIFIED!")
    print("Total images:", total_images)
    print("Location:", OUTPUT_DIR)
    print("READY FOR MODEL TRAINING!")
    print("====================================")

else:

    raise RuntimeError(
        "Dataset verification failed. Check Google Drive."
    )



========== VERIFYING DATASET ==========

DATASET CREATED AND VERIFIED!
Total images: 2786
Location: /content/drive/MyDrive/Dataset/Lung_Cancer_Augmented_Dataset_V4
READY FOR MODEL TRAINING!
